# Benchmark chirps — adaptive_v2 + modélisation seedée

Cette version teste le détecteur multibande `adaptive_v2` et passe la fréquence de pic du candidat à `process_full_spectrum()` pour initialiser la modélisation dans la bonne bande fréquentielle.

⚠️ Les paramètres `adaptive_v2` ont été optimisés sur le petit jeu actuel (8 WAV / 158 chirps). Ils restent expérimentaux jusqu'à validation sur davantage de WAV et des fichiers `no_chirp`.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
from tkinter import Tk, filedialog
import pandas as pd

from benchmark import run_benchmark


## Localiser le projet

In [ ]:
def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'bat_analysis').exists():
            return candidate
    raise FileNotFoundError('Racine bat_full_spectrum_processing introuvable')

PROJECT_ROOT = find_project_root()
analysis_py = PROJECT_ROOT / 'bat_analysis' / 'modelling.py'
print('Projet     :', PROJECT_ROOT)
print('Processing :', analysis_py)

## Choisir `bat_chirp_annotations.json`

In [ ]:
root = Tk()
root.withdraw()
root.attributes('-topmost', True)
annotation_json = filedialog.askopenfilename(
    title='Sélectionner bat_chirp_annotations.json',
    filetypes=[('JSON', '*.json'), ('Tous les fichiers', '*.*')],
)
root.destroy()
annotation_json = Path(annotation_json)
print('Annotations:', annotation_json)

## Paramètres expérimentaux actuels

Le canal général reste à 10 dB SNR. Un second canal limité à <=45 kHz utilise 9 dB pour récupérer les appels faibles et peu pentus. Le NMS est temps + fréquence pour conserver des composantes simultanées séparées.

In [ ]:
adaptive_v2_kwargs = {
    'slope_filter_mode': 'adaptive_v2',
    'snr_threshold_db': 10.0,
    'lowfreq_snr_threshold_db': 9.0,
    'percentile_q': 96.0,
    'fmin': 20000,
    'fmax': 150000,
    'n_fft': 512,
    'hop': 128,
    'min_blob_size': 10,
    'min_blob_height_hz': 5000.0,
    'general_max_blob_slope_hz_per_ms': -500.0,
    'lowfreq_max_hz': 45000.0,
    'lowfreq_min_blob_height_hz': 2000.0,
    'lowfreq_max_blob_slope_hz_per_ms': 0.0,
    'lowfreq_min_width_ms': 1.0,
    'echo_suppression_window_ms': 16.0,
    'echo_suppression_freq_window_hz': 25000.0,
}
adaptive_v2_kwargs

## Lancer le benchmark complet

In [ ]:
result = run_benchmark(
    annotation_json=annotation_json,
    analysis_py=analysis_py,
    detector_kwargs=adaptive_v2_kwargs,
    min_iou=0.05,
    max_center_error_ms=4.0,
    verbose=True,
)

## Résumé global

In [ ]:
pd.Series(result.summary)

## Comparaison aux baselines

In [ ]:
baseline_no_gate = {
    'detections': 141,
    'true_positives': 113,
    'false_positives': 28,
    'false_negatives': 45,
    'precision': 0.801418,
    'recall': 0.715190,
    'f1': 0.755853,
    'model_success': 113,
    'model_success_rate_on_tp': 1.0,
    'end_to_end_recall': 0.715190,
    'curve_median_abs_error_khz_mean': 2.275018,
    'curve_rmse_khz_mean': 2.787106,
    'curve_p95_khz_mean': 4.134084,
    'curve_coverage_mean': 0.848550,
}
keys = list(baseline_no_gate)
comparison = pd.DataFrame({
    'legacy_no_gate': pd.Series(baseline_no_gate),
    'adaptive_v2_seeded': pd.Series({k: result.summary.get(k) for k in keys}),
})
comparison['delta'] = comparison['adaptive_v2_seeded'] - comparison['legacy_no_gate']
comparison

## Résultats par WAV

In [ ]:
result.files

## Faux négatifs restants

In [ ]:
fn = result.chirps[result.chirps['failure_stage'].fillna('') == 'detection'].copy()
print(f'{len(fn)} FN')
fn

## Faux positifs restants

In [ ]:
fp = result.detections[result.detections['matched'] == False].copy()
print(f'{len(fp)} FP')
fp

## Erreurs de courbe les plus élevées

In [ ]:
curve_cols = [
    'relative_path', 'chirp_id', 'detector_branch', 'candidate_peak_freq_khz',
    'median_abs_error_khz', 'rmse_khz', 'p95_abs_error_khz', 'coverage'
]
result.chirps[result.chirps['model_success'] == True][[c for c in curve_cols if c in result.chirps.columns]].sort_values('rmse_khz', ascending=False).head(20)

## Sauvegarder les résultats

In [ ]:
output_dir = Path(annotation_json).parent / 'benchmark_results' / 'adaptive_v2_seeded'
result.save_csv(output_dir)
comparison.to_csv(output_dir / 'comparison_vs_legacy_no_gate.csv')
print('Résultats sauvegardés dans :', output_dir)